# Data Wrangling 2.3

In [ ]:
import math
import numpy as np
import pandas as pd

import psycopg2

import json

import csv

from datetime import datetime as dt

from IPython.display import display, HTML


In [ ]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [ ]:
cursor = connection.cursor()

In [ ]:
#
# function to run a select query and return rows in a pandas dataframe
# pandas puts all numeric values from postgres to float
# if it will fit in an integer, change it to integer
#

def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)
    

   # Lab: Data Cleansing - Data That Does Not Match Validation Rules

## The invalid sales date of 2021-17-14 will generate an error when we try to convert it from varchar to date

In [ ]:
connection.rollback()

query = """

select sale_date::date
from stage_3_sales


"""
cursor.execute(query)

            
    

## You try it - see if the product_id in line_items has valid numeric data

# Lab: Data Cleansing - Data That Does Not Match Lookup Tables

## product_id's that are not in the products table (we have to filter out the 'A' we just found)

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query  = """

with a as (select * from stage_3_line_items where product_id <> 'A')

select product_id::numeric
from a 
where product_id::numeric not in (select product_id from products)


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

## You try it - find customer_id's in the stage_3_sales table that are not in the stage_3_customers table

# Lab: Data Cleansing - Data That Violates Referential Integrity

## Find line items without a sales record

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query  = """

select *
from stage_3_line_items
where (store_id, sale_id) not in (select store_id, sale_id from stage_3_sales)


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

## You try it - find store_id's in the stage_3_sales table that are not in the stores table